
## @contextmanager 简化上下文管理
如果每次写上下文管理器都要定义一个类并实现 __enter__ 和 __exit__，代码会显得很臃肿。contextlib 模块提供的 @contextmanager 装饰器，可以让你用一个简单的生成器函数（yield）直接实现上下文管理器。
### 核心原理
- yield 之前的代码：相当于 __enter__（初始化资源）。
- yield 返回的值：相当于 as 后面的变量。
- yield 之后的代码：相当于 __exit__（清理资源）。

代码示例

In [1]:
from contextlib import contextmanager

@contextmanager
def open_my_file(file_path):
    # 1. 相当于 __enter__
    print(f"--- 正在打开文件: {file_path} ---")
    f = open(file_path, 'w', encoding='utf-8')

    try:
        # 2. 将资源传递给 as 后面的变量
        yield f
    finally:
        # 3. 相当于 __exit__（确保即便报错，也会执行清理）
        print("--- 正在自动关闭文件 ---")
        f.close()

# 使用它
with open_my_file('./11-contexlib-demo.txt') as file:
    file.write('Hello Contextlib!')
    print("文件写入中...")

--- 正在打开文件: ./11-contexlib-demo.txt ---
文件写入中...
--- 正在自动关闭文件 ---


### 如何在 __exit__ 中捕获并处理异常
无论是传统类写法，还是 contextlib 写法，你都可以选择捕获并消化（屏蔽）异常，或者让异常继续向上抛出。

① 在传统类中处理（通过返回值控制）
- __exit__(self, exc_type, exc_val, exc_tb) 方法接收三个异常参数：
- exc_type: 异常类型（如 ValueError）
- exc_val: 异常实例/错误信息
- exc_tb: 异常追踪栈对象（traceback）

💡 核心规则：如果 __exit__ 返回 True，Python 会吞掉（忽略）该异常，程序继续正常往下走；如果返回 False 或不返回（即 None），异常则会正常向上抛出。

In [ ]:
class ExceptionHandler:
    def __enter__(self):
        print("进入上下文...")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        # 检查是否有异常发生
        if exc_type is not None:
            print(f"🚨 捕获到异常类型: {exc_type}")
            print(f"💬 错误信息: {exc_val}")

            if exc_type is ZeroDivisionError:
                print("🛠️ 成功处理了除零错误，不向外抛出。")
                return True  # 返回 True，代表异常已被处理，不再向上抛出

        print("退出上下文（无异常或异常未处理）")
        return False  # 返回 False，未处理的异常会正常报错崩溃

② 在 contextlib 中处理（使用 try...except）在 @contextmanager 装饰的函数中，yield 语句就是“危险区域”。如果 with 块内部发生异常，该异常会在 yield 的位置被抛出。你只需要用标准的 try...except 包裹 yield 即可。

In [ ]:
from contextlib import contextmanager

@contextmanager
def safe_context():
    print("初始化...")
    try:
        yield
    except ZeroDivisionError:
        print("🛠️ contextlib 成功捕获并处理了除零错误！")
    finally:
        print("清理工作完成。")

# 测试
with safe_context():
    1 / 0  # 触发除零错误
print("程序未崩溃，继续向后执行。")